In [1]:
import pandas as pd
import numpy as np
import arviz as az
from jax import numpy as jnp
from jax.random import PRNGKey
import numpyro
from numpyro import distributions as dist
from numpyro import infer

pd.options.plotting.backend = "plotly"

from summer3.epi import CompartmentalModelODE, CategoryData, \
    strat_data_from_pandas, build_istate, dti_to_epoch

from tb_macro.constants import ALL_COMPARTMENTS, AGE_STRATA
from tb_macro.epi import get_base_model, add_infection_flows, \
    add_natural_history, add_health_system_flows, add_ageing_flows, \
    add_seeding, add_detection

In [2]:
epi_model, disease_state, age_strat, clin_strat, infect_strat = get_base_model()
add_infection_flows(epi_model, disease_state, age_strat.categories())
add_natural_history(epi_model, disease_state, clin_strat, infect_strat)
add_health_system_flows(epi_model, disease_state, clin_strat, infect_strat)
add_ageing_flows(epi_model, age_strat)
add_seeding(epi_model, disease_state)
add_detection(epi_model, disease_state, clin_strat)

In [3]:
age_strings = [str(a) for a in AGE_STRATA]
pop_data = pd.Series(index=age_strings, data=np.array([1000.0, 1000.0, 1000.0]))
base_pops = strat_data_from_pandas(pop_data, age_strat)
init_splits = [0.0] * len(ALL_COMPARTMENTS)
init_splits[ALL_COMPARTMENTS.index("mtb_naive")] = 1.0
pop_splits = [CategoryData(disease_state.categories(), jnp.array((init_splits)))]
epi_model.set_initial_population(base_pops, pop_splits)

In [4]:
def get_runner(epi_model):
    istate = build_istate(epi_model.cmap, epi_model.base_pops, epi_model.pop_splits)
    cmodel = CompartmentalModelODE(epi_model.cmap, epi_model.flows)
    runner = cmodel.get_runner(
        len(epi_model.times), dti_to_epoch(epi_model.times), True
    )
    return runner, istate

In [5]:
base_params = {
    "contact_rate": 0.25,
    "freq_dens_exponent": 1.0,
    "rel_sus_mtb_naive": 1.0,
    "rel_sus_contained": 0.0, 
    "rel_sus_cleared": 0.0, 
    "rel_sus_recovered": 0.0, 
    "contain": 0.0,
    "clearance_rate": 0.0,
    "breakdown_rate": 0.0,
    "progression": 1.0,
    "increase_infect": 1.0,
    "decrease_infect": 1.0,
    "clinical_development": 1.0,
    "clinical_regression": 1.0,
    "self_recovery": 0.2,
    "detection": 0.0,
    "treatment_recovery": 1.0,
    "treatment_relapse": 0.0,
    "seed_peak_time": 30.0,
    "seed_peak_rate": 0.01,
    "seed_duration": 10.0,
    "recent_detection_rate": 0.7,
    "passive_detection_shape": 0.1,
    "passive_detection_inflection": 40.0,
    "passive_detection_past_frac": 0.75,  
}
runner, istate = get_runner(epi_model)
results = epi_model.run(base_params)

Scatter
Gather
Gather
Gather
Scatter
Gather
Gather
Gather


W0331 14:02:16.196640  668797 cpp_gen_intrinsics.cc:74] Empty bitcode string provided for eigen. Optimizations relying on this IR will be disabled.


In [6]:
results["compartments"].sumcats(compartment=disease_state.categories()).to_pandas_df().plot()

In [ ]:
inf_target = (
    results["flows"]["infect_mtb_naive"]
    .sum(to_dims="time")
    .to_pandas_df()
    .rolling(7)
    .sum()[7:60:7]
)["data"]
inf_target_fuzzy = inf_target * np.exp(
    np.random.normal(scale=0.01, size=len(inf_target))
)
inf_target_fuzzy.plot()

In [ ]:
def get_derived_results(params):
    results = epi_model.run(params)  # runner.run(istate.data, params)
    inf_flow = results["flows"]["infect_mtb_naive"]
    weekly_target = (
        inf_flow.sum(to_dims="time").rolling(7, jnp.sum).query(time=inf_target.index)
    )
    return weekly_target

In [ ]:
priors = {
    "contact_rate": dist.Uniform(0.001, 0.5),
}

In [ ]:
def model():
    params = base_params | {k: numpyro.sample(k, v) for k, v in priors.items()}
    weekly_modelled = get_derived_results(params)

    ll = dist.Poisson(inf_target_fuzzy.to_numpy()).log_prob(weekly_modelled.data)
    numpyro.factor("ll", ll)

In [ ]:
kernel = infer.NUTS(model)
mcmc = infer.MCMC(kernel, num_warmup=200, num_samples=200, num_chains=4)
k = PRNGKey(0)
mcmc.run(k)

In [ ]:
idata = az.from_numpyro(mcmc)

In [ ]:
az.summary(idata)

In [ ]:
az.plot_posterior(idata)

In [ ]:
az.plot_trace(idata, compact=False)